# Validating a fit

The other inference notebook asks whether a fit *converges*. This one asks a
harder question: whether the fitted model is any **good** — and answers it
without reusing the arithmetic that produced it.

That distinction is not pedantry. It exists because of one specific
cancellation, which this notebook demonstrates rather than describes:

> A fit made with a compensator 20% too small inflates the intensity, and then
> rescaling the events through that *same* broken integral gives unit-rate gaps.
> The goodness-of-fit test passes. The worse the compensator, the more exactly
> the fit compensates for it.

Four questions, four tools:

| Question | Tool |
|---|---|
| Is the compensator itself right? | `compensator_agreement` |
| Did events arrive in the right amounts, where and when? | `cell_residuals` |
| Is the model worth having at all? | `compare_with_baseline` |
| Would it have been any use prospectively? | `rolling_origin` |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import hawkes_package as hp
from hawkes_package.inference import (
    ConstrainedPrior,
    ExponentialLogLikelihood,
    History,
    IndependentPrior,
    LogNormal,
    exponential_model,
    fit_smc,
    ks_exponential,
    residuals,
)
from hawkes_package.inference.validation import (
    cell_residuals,
    compare_with_baseline,
    compensator_agreement,
    rolling_origin,
)

plt.rcParams["figure.figsize"] = (9, 3.2)

## The data and the fit

A linear Hawkes process with an exponential kernel, observed on a fixed
**horizon** rather than to a fixed count. That choice matters for what follows:
`simulate(k)` ends the window exactly at the k-th event, which conditions on an
event having just happened and makes the expected count exceed the observed one.
A real property of the stopping rule, not a defect — but it puts a bias into
every residual mean, and here we want those centred.

In [ ]:
TRUTH = np.array([2.0, 0.5, 1.0])  # mu, alpha, beta
HORIZON = 100.0

process = hp.ExponentialHawkes(TRUTH, rng=7)
process.simulate_until(HORIZON)
history = History.from_events(process.events, end=HORIZON)

model = exponential_model()
likelihood = ExponentialLogLikelihood(model)
prior = ConstrainedPrior(
    IndependentPrior((LogNormal(0.5, 1.0), LogNormal(-1.0, 1.0), LogNormal(0.0, 1.0))),
    model.support,
)

smc = fit_smc(likelihood, prior, history, blocks=5, n_particles=256, rng=0)
theta = smc.cloud.mean()

print(f"{history.n_events} events on [0, {HORIZON:.0f}]")
print(smc.cloud.summary())

## The diagnostic that already existed

Time rescaling: map the events through their own compensator and the gaps should
be independent `Exp(1)`. It catches a wrong kernel, a missing background and a
compensator computed too small, all at once.

In [ ]:
gaps = residuals(likelihood, theta, history)
result = ks_exponential(gaps)
print(f"mean gap {gaps.mean():.4f} (should be 1), KS p = {result.pvalue:.4g}")

## Why that is not enough

`residuals` takes its compensator from the likelihood it is handed. For checking
two implementations of one model against each other that is exactly right — a
bug in one cannot pass the test and fail the fit. For *validation* it is exactly
backwards.

Here is a likelihood whose integral is deliberately 20% short, which is what a
quadrature rule too coarse for its integrand actually does. Fitting against it
raises the excitation until the smaller penalty balances the log-sum, so the
fitted `alpha` comes out well above the truth.

In [ ]:
class ShortCompensator(ExponentialLogLikelihood):
    """A compensator 20% too small -- a systematic under-integration."""

    def compensator(self, theta, history, times):
        return 0.8 * super().compensator(theta, history, times)


broken = ShortCompensator(model)

# The parameter a too-small integral talks you into.
inflated = theta.copy()
inflated[1] = 0.9

through_broken = ks_exponential(residuals(broken, inflated, history))
through_honest = ks_exponential(residuals(likelihood, inflated, history))

print(f"residuals via the BROKEN compensator: p = {through_broken.pvalue:.4g}")
print(f"residuals via an HONEST  compensator: p = {through_honest.pvalue:.4g}")

The broken compensator gives a clean pass on a model that is visibly wrong. The
errors cancel: the intensity is too large by exactly the factor the integral is
too small by, so the rescaled gaps come out unit-rate anyway.

`compensator_agreement` is what notices. It integrates the simulator's own
intensity hook on a dense uniform grid — a different quadrature family, and a
code path that shares only the parameter vector with the likelihood it is
checking.

In [ ]:
print(f"agreement, honest likelihood: {compensator_agreement(likelihood, theta, history):.3g}")
print(f"agreement, broken likelihood: {compensator_agreement(broken, inflated, history):.3g}")

Around `1e-5` is two quadrature rules resolving the same integrand — the
integrand jumps at every event and a uniform grid cannot place a node there, so
some disagreement is expected. `0.2` is the defect, reported at its actual size.

**Read this number before reading any goodness-of-fit test below it.**

## Where and when, not only how often

Time rescaling collapses the window before it starts, so when it rejects it says
only that something is wrong — never which part. Cell residuals compare the
observed count in each cell against the integral of the intensity over it. Under
the true model that count is Poisson, so the standardised residual is roughly
standard normal.

In [ ]:
at_fit = cell_residuals(likelihood, theta, history, n_cells=20)
at_wrong = cell_residuals(likelihood, inflated, history, n_cells=20)

print(at_fit.summary())
print()
print(at_wrong.summary())

In [ ]:
centres = 0.5 * (at_fit.edges[:-1] + at_fit.edges[1:])
fig, ax = plt.subplots()
width = 0.4 * (centres[1] - centres[0])
ax.bar(centres - width / 2, at_fit.standardised, width, label="at the fit")
ax.bar(centres + width / 2, at_wrong.standardised, width, label="at alpha = 0.9")
for level in (-2, 2):
    ax.axhline(level, color="grey", lw=0.8, ls="--")
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("time")
ax.set_ylabel("standardised residual")
ax.set_title("cell residuals: two standard deviations is the usual eyebrow")
ax.legend()
plt.show()

At the fit the residuals scatter around zero. At the inflated excitation they sit
below it almost everywhere — the model predicts more events than arrived, in
every part of the window rather than in one place.

Cells too empty to standardise come back as `nan` rather than as a huge number:
one stray event in a cell expecting 0.01 standardises to 10 and would dominate
every real signal beside it.

## Is the model worth having

A model can pass every check above and still predict no better than a constant
rate. The bar is the homogeneous Poisson process at its **own** maximum
likelihood — not at some convenient rate, because a baseline handed a bad
parameter is worse than no baseline: beating it reads as evidence.

In [ ]:
print(compare_with_baseline(likelihood, theta, history).summary())

Now the same comparison on data with the excitation switched off. The Hawkes
model has nothing to find, and should say so.

In [ ]:
flat_truth = np.array([2.0, 1e-6, 1.0])
flat = hp.ExponentialHawkes(flat_truth, rng=7)
flat.simulate_until(HORIZON)
flat_history = History.from_events(flat.events, end=HORIZON)

print(compare_with_baseline(likelihood, flat_truth, flat_history).summary())

## Would it have been any use prospectively

Everything so far is in-sample: the model is judged against the events that chose
its parameters. `rolling_origin` refits at a series of origins and scores only
what came after each.

The fitter is handed the prefix and nothing else — that is the one guarantee the
function provides. The *intensity* still sees the whole past, because a Hawkes
intensity depends on history by construction and withholding it would not be a
stricter test but a different model.

**This section needs more data than the rest of the notebook, and the reason is
worth stating.** Five blocks of fifty events cannot separate a branching ratio of
0.5 from a constant rate: on the hundred-unit window above, even the *true*
parameters score negative skill at three of five origins. That is the backtest
being underpowered, not the model being useless — and it is exactly the kind of
result that gets misread as the second. Four hundred units is enough for the
answer to stabilise.

In [ ]:
long_process = hp.ExponentialHawkes(TRUTH, rng=7)
long_process.simulate_until(400.0)
long_history = History.from_events(long_process.events, end=400.0)

print(f"{long_history.n_events} events on [0, 400]")

In [ ]:
def fit_on(prefix):
    """Refit from scratch on everything up to an origin."""
    return fit_smc(likelihood, prior, prefix, blocks=3, n_particles=128, rng=0).cloud.mean()


backtest = rolling_origin(likelihood, long_history, fit_on, origins=np.linspace(80.0, 320.0, 5))
print(backtest.summary())

In [ ]:
origins = [s.origin for s in backtest.scores]
skill = [s.skill for s in backtest.scores]

fig, ax = plt.subplots()
ax.bar(origins, skill, width=32.0, color=["tab:green" if s > 0 else "tab:red" for s in skill])
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("origin")
ax.set_ylabel("skill (nats per event)")
ax.set_title("out-of-sample skill against a constant rate, per origin")
plt.show()

Positive skill means the fit predicted the next block better than the best
constant rate would have. Not necessarily at every origin — a block can be quiet
by chance — which is why the aggregate is what carries the claim and why the
window above had to be long enough for that aggregate to settle.

## What each of these is for

| Tool | Catches |
|---|---|
| `compensator_agreement` | a compensator wrong in a way the residuals cannot see |
| `cell_residuals` | a model wrong in one part of the window |
| `compare_with_baseline` | a model that fits fine and predicts nothing |
| `rolling_origin` | a model that only works in hindsight |

None of them replaces the others, and `compensator_agreement` comes first:
if the integral is wrong, every test built on it is reporting a number about the
wrong model.